# AutoCIM-Agent — 시연 노트북

**CIM(Compute-In-Memory) 반도체 HW-SW 공동설계를 위한 LangGraph 기반 7-에이전트 파이프라인**

이 노트북은 발표 시연용입니다. 아래 순서로 보여줍니다.

| # | 내용 | 예상 시간 |
|---|---|---|
| 1 | 에이전트 그래프 구조 | 즉시 |
| 2 | 1 iteration 실제 실행 (실제 QAT 학습 포함) | 약 12초 |
| 3 | **HITL 인터럽트 → 재개** | 12초 + 즉시 |
| 4 | 실측 결과 (미리 측정해둔 값) | 즉시 |
| 5 | HTML 대시보드 | 즉시 |

> ⚠️ **이 노트북의 학습 규모는 시연용(64장/1에폭)입니다.** 여기서 나오는 정확도(약 0.17)는 아무 의미가 없습니다.
> 실제 측정값은 **4096장 3시드**로 따로 돌린 결과이며 §4에서 보여줍니다.

## 0. 워밍업 — **발표 시작 전에 미리 실행하세요**

이 셀만 미리 돌려두면 이후 모든 셀이 빠르게 실행됩니다 (모델/데이터셋 캐시 로드).

In [1]:
import os
import sys
from pathlib import Path

# 노트북이 notebooks/ 안에 있어도 프로젝트 루트 기준으로 동작하게 맞춤
ROOT = Path.cwd()
if not (ROOT / "graph.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# 시연 규모: 셀 하나가 몇 초 안에 끝나도록. 실제 측정은 4096장으로 별도 수행(§4).
os.environ["AUTOCIM_QAT_TRAIN_SIZE"] = "64"
os.environ["AUTOCIM_QAT_VAL_SIZE"] = "32"
os.environ["AUTOCIM_QAT_TEST_SIZE"] = "64"
os.environ["AUTOCIM_QAT_MAX_EPOCHS"] = "1"

import json
import time

from langgraph.types import Command

from graph import build_graph
from main import build_initial_state
from middleware import register_hw_config
from schemas.config import HWConfig
from nodes.evaluator import MAX_RETRY_LIMIT
from tools.qat import get_qat_backend, get_layer_groups

HW = HWConfig(**json.load(open("examples/hw_configs/default_cim_v1_128x128.json")))
register_hw_config(HW)

t = time.perf_counter()
get_qat_backend("resnet18")          # 모델 + CIFAR10 캐시 워밍업
n_stages = len(get_layer_groups("resnet18"))
print(f"워밍업 완료: {time.perf_counter() - t:.1f}초")
print(f"대상 하드웨어: {HW.crossbar_rows}x{HW.crossbar_cols} 크로스바, "
      f"ADC {HW.adc_bits}bit, DAC {HW.dac_bits}bit, 타일 {HW.num_tiles}개")
print(f"resnet18 탐색 대상 stage: {n_stages}개 → 탐색 공간 {2 * n_stages}차원")

워밍업 완료: 2.2초
대상 하드웨어: 128x128 크로스바, ADC 8bit, DAC 4bit, 타일 16개
resnet18 탐색 대상 stage: 10개 → 탐색 공간 20차원


---
## 1. 에이전트 그래프 구조

`graph.py`가 조립한 **실제 LangGraph 그래프**입니다. 아래 출력은 손으로 그린 게 아니라
컴파일된 그래프에서 직접 추출한 것입니다.

In [2]:
graph = build_graph()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	tuner(tuner)
	mapper(mapper)
	profiler(profiler)
	verifier(verifier)
	evaluator(evaluator)
	precision_verifier(precision_verifier)
	hitl_human_approval(hitl_human_approval)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	evaluator -. &nbsp;HITL Interrupt&nbsp; .-> hitl_human_approval;
	evaluator -. &nbsp;Re-plan with History&nbsp; .-> planner;
	evaluator -. &nbsp;Converged (Done)&nbsp; .-> precision_verifier;
	hitl_human_approval -. &nbsp;Concluded (low confidence)&nbsp; .-> __end__;
	hitl_human_approval -. &nbsp;Re-plan&nbsp; .-> planner;
	mapper --> verifier;
	planner --> tuner;
	precision_verifier -. &nbsp;Converged (Done)&nbsp; .-> __end__;
	precision_verifier -. &nbsp;HITL Interrupt&nbsp; .-> hitl_human_approval;
	precision_verifier -. &nbsp;Re-plan with History&nbsp; .-> planner;
	profiler --> verifier;
	tuner --> mapper;
	tuner --> profiler;
	verifier -->

```mermaid
graph TD;
    START([__start__]) --> planner
    planner["@planner - 탐색/대리모델"] --> tuner
    tuner["@tuner - 실제 PyTorch QAT"] --> mapper
    tuner --> profiler
    mapper["@mapper - NoC 매핑"] --> verifier
    profiler["@profiler - 에너지/지연 물리 시뮬"] --> verifier
    verifier["@verifier - IR-drop/노이즈 검증"] --> evaluator
    evaluator["@evaluator - 수렴 판정/Pareto rank"] -- "Re-plan" --> planner
    evaluator -- "HITL Interrupt" --> hitl["hitl_human_approval"]
    evaluator -- "Converged" --> precision["@precision_verifier - 2단계 정밀 재검증"]
    precision -- "Converged" --> END([__end__])
    precision -- "HITL Interrupt" --> hitl
    hitl -- "Re-plan" --> planner
    hitl -- "낮은 신뢰도 종결" --> END
```

**요점**: `@mapper`와 `@profiler`는 서로 독립이라 **병렬 fan-out**으로 실행되고 `@verifier`에서 fan-in 됩니다.
두 노드가 동시에 상태에 쓰기 때문에, 각 노드는 `metrics_store`의 **자기 네임스페이스에만** 쓰도록
강제됩니다(`namespaced_metrics_reducer`) — 그렇지 않으면 경쟁 조건으로 서로의 결과를 덮어씁니다.

---
## 2. 1 iteration 실제 실행

`@planner`가 후보 제안 → `@tuner`가 **실제 PyTorch 그래디언트 하강으로 QAT 학습** →
`@mapper`/`@profiler`가 물리 시뮬레이션 → `@verifier`가 IR-drop/노이즈 검증 → `@evaluator`가 판정.

실행 시간의 대부분은 **로컬 Ollama LLM**이 rationale을 생성하는 시간입니다.

In [3]:
state = build_initial_state("resnet18", HW, target_accuracy=0.99)
cfg = {"configurable": {"thread_id": f"demo-iter-{int(time.time())}"}}

t, done = time.perf_counter(), False
for chunk in graph.stream(state, config=cfg, stream_mode="updates"):
    for node, update in chunk.items():
        if node == "planner":
            d = update["planner_decisions"][0]
            print(f"[@planner  ] {d['search_tag']}  (LLM 사용={d['used_llm']})")
            print(f"             rationale: {str(d['rationale'])[:90]}...")
        elif node == "tuner":
            print(f"[@tuner    ] 실제 QAT 학습 → accuracy="
                  f"{update['metrics_store']['tuner']['data']['accuracy']}")
        elif node == "mapper":
            m = update["metrics_store"]["mapper"]["data"]
            print(f"[@mapper   ] 필요 타일={m['tiles_needed']}개  NoC 지연={m['noc_latency_ms']}ms")
        elif node == "profiler":
            p = update["metrics_store"]["profiler"]["data"]
            print(f"[@profiler ] 에너지={p['energy_pj']:,.0f} pJ")
        elif node == "verifier":
            v = update["metrics_store"]["verifier"]["data"]
            print(f"[@verifier ] 물리검증={v['is_converged']}  IR-drop={v['ir_drop_error_pct']}%  "
                  f"노이즈마진={v['noise_margin_db']}dB")
        elif node == "evaluator":
            print(f"[@evaluator] 수렴={update.get('is_converged')}  (재시도 {update.get('retry_count')})")
            done = True   # 1 iteration(= @evaluator 도달)까지만 보여주고 중단
    if done:
        break

print(f"\n총 {time.perf_counter() - t:.1f}초")
print("※ accuracy가 낮은 건 시연용 64장 학습이기 때문입니다 — 실제 측정은 §4 참고")

[@planner  ] LHS warm-up candidate 1/15  (LLM 사용=True)
             rationale: The initial candidate from the Latin Hypercube Sampling (LHS) warm-up suggests a low weigh...


[@tuner    ] 실제 QAT 학습 → accuracy=0.1719
[@mapper   ] 필요 타일=699개  NoC 지연=0.000368ms
[@profiler ] 에너지=2,065,581 pJ
[@verifier ] 물리검증=True  IR-drop=0.064%  노이즈마진=47.16dB
[@evaluator] 수렴=False  (재시도 1)

총 9.3초
※ accuracy가 낮은 건 시연용 64장 학습이기 때문입니다 — 실제 측정은 §4 참고


---
## 3. HITL (Human-in-the-Loop) 인터럽트 — **이 프로젝트의 핵심 경로**

이 파이프라인에서 HITL은 예외 처리가 아니라 **설계상 핵심 경로**입니다.
탐색·근사 모델을 완전히 신뢰할 수 없다는 전제 위에 만들어졌기 때문에,
`__end__`는 "정답 확정"이 아니라 **"사람이 검토할 후보가 준비됨"**을 뜻합니다.

LangGraph v0.2+ 표준인 **동적 `interrupt()`** 만 사용합니다
(`compile()`에 `interrupt_before`를 절대 주지 않음).

아래는 **이미 2번 실패한 세션을 이어받는** 상황입니다 — 체크포인트에서 재개하는 실제 시나리오.

In [4]:
state = build_initial_state("resnet18", HW, target_accuracy=0.99)
state["retry_count"] = MAX_RETRY_LIMIT - 1        # 이미 2회 실패한 세션을 이어받음
cfg = {"configurable": {"thread_id": f"demo-hitl-{int(time.time())}"}}

t = time.perf_counter()
result = graph.invoke(state, config=cfg)
print(f"실행 {time.perf_counter() - t:.1f}초 → 그래프가 멈췄습니다\n")

payload = result["__interrupt__"][0].value
print("=== 사람에게 전달된 인터럽트 페이로드 ===")
for k in ("reason", "iteration_count", "retry_count", "hw_spec_id", "target_accuracy"):
    print(f"  {k:<16}: {payload.get(k)}")

실행 8.1초 → 그래프가 멈췄습니다

=== 사람에게 전달된 인터럽트 페이로드 ===
  reason          : Convergence stalled -- researcher review requested
  iteration_count : 1
  retry_count     : 3
  hw_spec_id      : cim_v1_128x128
  target_accuracy : 0.99


### 재개 — 두 가지 선택지

연구자는 두 가지로 재개할 수 있습니다.

| 선택 | `Command(resume=...)` | 결과 |
|---|---|---|
| 탐색 범위를 좁혀 재시도 | `{"new_bounds": {...}}` | `@planner`로 돌아가 계속 탐색 |
| **지금까지의 최선으로 종결** | `{"conclude_with_best": True}` | 바로 `__end__`, **낮은 신뢰도로 표시** |

아래는 두 번째 경로입니다. `is_converged=True`가 되지만 `low_confidence_conclusion=True`가
함께 붙어서, **"목표를 진짜 만족한 수렴"과 "포기하고 고른 최선"을 절대 혼동하지 않게** 합니다.

In [5]:
t = time.perf_counter()
final = graph.invoke(Command(resume={"conclude_with_best": True}), config=cfg)
print(f"재개 {time.perf_counter() - t:.2f}초\n")
print(f"  is_converged              : {final.get('is_converged')}")
print(f"  low_confidence_conclusion : {final.get('low_confidence_conclusion')}   ← 검토 필요 표시")
print(f"  iteration_count           : {final.get('iteration_count')}")
print(f"  평가한 후보 수            : {len(final.get('candidate_history', []))}")
print("\n→ 이 표시가 붙은 결과는 CLI 최종 출력과 대시보드 모두에서 '검토 필요'로 구분 표시됩니다.")

재개 0.00초

  is_converged              : True
  low_confidence_conclusion : True   ← 검토 필요 표시
  iteration_count           : 1
  평가한 후보 수            : 1

→ 이 표시가 붙은 결과는 CLI 최종 출력과 대시보드 모두에서 '검토 필요'로 구분 표시됩니다.


---
## 4. 실측 결과 — **여기가 진짜 수치입니다**

아래는 시연용 64장이 아니라 **CIFAR10 4096장 · 3시드**로 별도 측정한 결과입니다.

이 프로젝트가 실제로 한 일은 "탐색이 최적해를 찾았다"가 아니라,
**실제로 돌려봤더니 물리 모델이 틀려 있었고 그걸 찾아 고쳤다**는 쪽입니다.

In [6]:
def load(path):
    try:
        return json.load(open(path))
    except FileNotFoundError:
        print(f"(없음: {path} — tools/baseline_compare.py 로 재생성 가능)")
        return None

v2 = load("baseline_compare_v2_results.json")
v3 = load("baseline_compare_v3_snap_results.json")
opt = load("optuna_compare_v2_results.json")

if v2 and v3:
    b = v3["baseline"]["summary"]
    s2, s3 = v2["search"]["summary"], v3["search"]["summary"]
    base_e = b["energy_pj"]["mean"]
    print("=== 균일 베이스라인 vs 탐색 (resnet18, 4096장, 3시드) ===")
    print(f"  베이스라인    : 정확도 {b['accuracy']['mean']:.3f}   에너지 {base_e:>12,.0f} pJ")
    print(f"  탐색(스냅 전) : 정확도 {s2['accuracy']['mean']:.3f}   에너지 "
          f"{s2['energy_pj']['mean']:>12,.0f} pJ  → 절감 {100*(1-s2['energy_pj']['mean']/base_e):.1f}%")
    print(f"  탐색(스냅 후) : 정확도 {s3['accuracy']['mean']:.3f}   에너지 "
          f"{s3['energy_pj']['mean']:>12,.0f} pJ  → 절감 {100*(1-s3['energy_pj']['mean']/base_e):.1f}%")
    print("\n  ※ 프루닝을 크로스바 타일 경계로 스냅 → 정확도 손실 0으로 에너지만 6.3% 추가 절감")

if opt:
    print("\n=== 자체 탐색 vs Optuna (동일 파이프라인·동일 정지조건, 3시드) ===")
    for k, v in opt.items():
        sm = v["summary"]
        print(f"  {k:<32} 수렴 {sm['n_converged']}/3   정확도 {sm['accuracy']['mean']:.3f}")

=== 균일 베이스라인 vs 탐색 (resnet18, 4096장, 3시드) ===
  베이스라인    : 정확도 0.901   에너지    2,285,274 pJ
  탐색(스냅 전) : 정확도 0.823   에너지    2,084,137 pJ  → 절감 8.8%
  탐색(스냅 후) : 정확도 0.824   에너지    1,952,837 pJ  → 절감 14.5%

  ※ 프루닝을 크로스바 타일 경계로 스냅 → 정확도 손실 0으로 에너지만 6.3% 추가 절감

=== 자체 탐색 vs Optuna (동일 파이프라인·동일 정지조건, 3시드) ===
  multi_objective (this project)   수렴 1/3   정확도 0.691
  optuna (NSGA-II, direct)         수렴 0/3   정확도 0.524


### 실제로 발견한 물리 모델 버그 3건

전부 **실제로 돌려보지 않으면 유닛 테스트로는 잡히지 않는** 종류입니다.

| # | 버그 | 영향 |
|---|---|---|
| 1 | SRAM 크로스바 MAC 에너지가 **정확히 0.0** | NeuroSim C++ 소스까지 추적 — 저항성 셀 모델이 디지털 SRAM에 구조적으로 부적용 |
| 2 | 물리 모델이 **레이어 형상을 아예 안 봄** | resnet18 **69.9배**, mobilenet_v2 **37.3배** 과소평가. 배율이 아키텍처마다 달라 **교차 비교가 왜곡** |
| 3 | `column_pruning_ratio`가 에너지에 **무영향** | 탐색 2개 차원 중 하나가 **보상 없는 순수 손해**였음 |

그리고 고친 뒤 **전부 재측정했고, 기존 결론이 그대로 유지**됐습니다.
"고치니 유리해졌다"가 아니라 **"고쳐도 주장이 살아남았다"** 입니다.

---
## 5. HTML 대시보드

`tools/dashboard.py`가 세션 상태를 그대로 렌더링합니다.
위 §3에서 만든 **낮은 신뢰도 종결** 세션이라, 경고 배지가 함께 표시됩니다.

In [7]:
from IPython.display import HTML, display
from tools.dashboard import render_dashboard_html

html = render_dashboard_html(final)
with open("notebooks/demo_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"저장: notebooks/demo_dashboard.html ({len(html):,} bytes)")
display(HTML(html))

저장: notebooks/demo_dashboard.html (13,431 bytes)


iter반복,search phase탐색 단계,accuracy정확도,energy_pj,latency_ms,pareto_rank,converged수렴 여부,rationale근거
1,LHS warm-up candidate 1/15,0.1719,2065580.60,0.000,1,no아니요,"The initial candidate from the Latin Hypercube Sampling (LHS) warm-up suggests a low weight bit configuration for most layers, with `weight_bits=2` and no column pruning (`column_pruning_ratio=0.0`). This is reasonable given the hardware bounds of `dac_bits=4`, as it allows for some flexibility in quantization without exceeding the DAC's bit limit. The `fc` layer also follows this pattern, maintaining a low weight bit configuration. Since there are no recent failures and the metrics are not yet available, we can proceed with these settings to explore the performance space.초기 후보는 라틴 하이퍼 kup스 샘플링 (LHS)에서 추천한 것으로, 대부분의 레이어에 대해 `weight_bits=2`와 열 전결 비율이 0.0인 설정을 제안합니다. 이는 하드웨어 제약 사항 (`dac_bits=4`) 내에서 몇 가지 유연성을 제공하면서 DAC의 비트 제한을 초과하지 않도록 설계되었습니다. `fc` 레이어도 이러한 패턴을 따르며, 낮은 weight bit 설정을 유지합니다. 최근 실패 사례가 없으며 메트릭이 아직 없는 점을 고려하면 이러한 설정으로 성능 공간을 탐색할 수 있습니다."


---
## 정리

- **LangGraph 멀티에이전트 + 동적 `interrupt()` HITL + 구조화 출력(forced tool-calling)** —
  1단계 교육에서 배운 내용을 실제 연구 문제에 적용
- **LLM은 수치를 정하지 않습니다.** 로컬 모델이 정확도 문제에서 비트 수를 거꾸로 추론하는 걸
  실측으로 발견한 뒤, 역할을 rationale 설명과 이상 징후 플래깅으로 좁혔습니다
- **정직성이 설계 원칙**: 근사/실제 모델 구분 표시, 낮은 신뢰도 종결 표시, 캘리브레이션 출처 배지
- 479개 테스트 · CI · Docker 실제 소자 모델 연동

자세한 내용은 [`README.md`](../README.md), 개념 배경은 [`docs/concepts.md`](../docs/concepts.md).